In [1]:
import os
os.environ['HF_ENDPOINT'] = "https://hf-mirror.com"

## (1) Load Model & Weights from HuggingFace

In [2]:
from model import Mamba
from helper import load_from_cache, generate, prefill, step_fn

base_path="/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("load done")

/root/proj/spu/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


load done


In [3]:
def generate_demo(prompt, gen_len=10, seed=42):
    input_ids = tokenizer.encode(prompt, return_tensors='jax')
    output_ids = generate(model, params, input_ids, gen_len, seed=seed)
    print(prompt, tokenizer.decode(output_ids[0]), sep='')

In [4]:
generate_demo('Mamba is the')

Mamba is the highest mountain in the province, and also the highest


In [5]:
generate_demo('The meaning of life is ')

The meaning of life is ʻinērʻ (ʻ


In [6]:
generate_demo('def reverse_string(')

def reverse_string(self, prefix):
    """
    Helper


In [7]:
generate_demo('The meaning of life is ', seed=114514)

The meaning of life is 
"a state of being and the state of


## (2) SPU

### 2.1: 定义simulator

> 教程视频中用了一堆的`spu_pb2`，但现在的spu中根本没有这个模块，根据成员推测，尝试换成`libspu`

In [8]:
import spu.utils.simulation as spsim
import spu.libspu as libspu

sim = spsim.Simulator.simple(2,libspu.ProtocolKind.CHEETAH, libspu.FieldType.FM64)
# sim_aby = spsim.Simulator.simple(2,libspu.ProtocolKind.ABY3, libspu.FieldType.FM64)

In [9]:
# define cheetah config with pphlo trace and profile on
config_che = libspu.RuntimeConfig(
    protocol = libspu.ProtocolKind.CHEETAH,
    field = libspu.FieldType.FM32,
    fxp_fraction_bits = 10,
    # enable_pphlo_trace = True,
    # enable_pphlo_profile = True,
)

### 2.2: 定义运行函数

In [10]:
import jax
import jax.numpy as jnp
from functools import partial

# gen_spu = jax.jit(generate, static_argnames=['model','n_tokens_to_gen','sample','top_k'])

@partial(jax.jit, static_argnames=['n_tokens_to_gen','sample','top_k'])
def gen_spu(params, input_ids, n_tokens_to_gen: int = 5,
             sample: bool = True, top_k: int = 40, seed: int = 42):
    key = jax.random.PRNGKey(seed)
    next_token_logits, states = prefill(model, params, input_ids)

    generated = jnp.zeros((input_ids.shape[0], n_tokens_to_gen), dtype=input_ids.dtype)

    for i in range(n_tokens_to_gen):
        if top_k is not None:
            values, _ = jax.lax.top_k(next_token_logits, k=top_k)
            kth_values = values[:, -1:]
            next_token_logits = jnp.where(next_token_logits < kth_values, -1e9, next_token_logits)

        probs = jax.nn.softmax(next_token_logits, axis=-1)

        if sample:
            key, subkey = jax.random.split(key)
            next_id = jax.random.categorical(subkey, jnp.log(probs + 1e-9), axis=-1)
        else:
            next_id = jnp.argmax(probs, axis=-1)

        generated = generated.at[:, i].set(next_id)

        next_token_logits, states = step_fn(model, params, next_id, states)

    return generated

### 2.3: 运行密态程序

In [11]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
# output_ids = spsim.sim_jax(sim, gen_spu)(
#     model, params, input_ids, n_tokens_to_gen = 10, sample = False, top_k = None, seed = 42
# )
output_ids = gen_spu(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is in a sense a kind


In [12]:
# prompt = "Python is"
# input_ids = tokenizer.encode(prompt, return_tensors='jax')
# output_ids = spsim.sim_jax(sim, gen_spu)(
#     params, input_ids
# )
# # output_ids = gen_spu(model, params, input_ids)
# print(prompt, tokenizer.decode(output_ids[0]), sep='')

20′11″后，输出了一坨屎
```plaintext
Python is�wg Tak concentrating
```

能观察到各个阶段的动静，先是大量的none夹着一些interleave和pack_lwes，这个阶段CPU不会吃满
```plaintext
[2026-03-31 11:04:53.484] [info] [cheetah_dot.cc:475] 1@16x1x2 => 16x1x2 Recv 0.176 MiB, Response 0.122 MiB Pack 0 ms (none)
```

然后进入计算阶段，这里的时间相对较长
```plaintext
[2026-03-31 11:13:17.542] [info] [cheetah_dot.cc:475] 1@1x1536x80 => 1x1024x8 Recv 0.353 MiB, Response 0.243 MiB Pack 43.256 ms (pack_lwes)
[2026-03-31 11:13:17.579] [info] [cheetah_dot.cc:475] 1@1x48x1536 => 1x32x256 Recv 0.353 MiB, Response 0.243 MiB Pack 21.342 ms (interleave)
```

## (3) SPU下验证性能
### 3.1 定义emulator

In [13]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS

# note: in MULTIPROCESS mode, bandwidth and latency doesn't work
# emulation.CLUSTER_ABY3_3PC is a hard-coded string
# we copied it to current folder
emulator = emulation.Emulator(
    "3pc.json",
    mode,
    bandwidth=100,
    latency = 10
)

emulator.up()

[2026-04-01 09:47:35,253]-[INFO]-[emulation.py:112]: Start multiprocess cluster...
[2026-04-01 09:47:35,795] [ForkServerProcess-5] Starting grpc server at 127.0.0.1:61924
[2026-04-01 09:47:35,795] [ForkServerProcess-2] Starting grpc server at 127.0.0.1:61921
[2026-04-01 09:47:35,795] [ForkServerProcess-4] Starting grpc server at 127.0.0.1:61923
[2026-04-01 09:47:35,795] [ForkServerProcess-1] Starting grpc server at 127.0.0.1:61920
[2026-04-01 09:47:35,803] [ForkServerProcess-3] Starting grpc server at 127.0.0.1:61922
[2026-04-01 09:47:37,338] [ForkServerProcess-3] Run : builtin_spu_init at node:2
[2026-04-01 09:47:37,338] [ForkServerProcess-1] Run : builtin_spu_init at node:0
[2026-04-01 09:47:37,339] [ForkServerProcess-2] Run : builtin_spu_init at node:1
I0401 09:47:37.357910 11511     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61931.
W0401 09:47:37.357927 11511     0 external/brpc~/src/brpc/server.cpp:120

In [14]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
# output_ids = spsim.sim_jax(sim, gen_spu)(
#     model, params, input_ids, n_tokens_to_gen = 10, sample = False, top_k = None, seed = 42
# )
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu)(s_params, s_input_ids)
print(result)
# print(prompt, tokenizer.decode(result[0]), sep='')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
[2026-04-01 09:47:38,606] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-01 09:47:38,650] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-01 09:47:38,653] [ForkServerProcess-4] Run : make_shares at node:3


[2026-04-01 09:47:38.655] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


[2026-04-01 09:47:41,818] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-01 09:47:41,821] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-01 09:47:41,826] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-01 09:47:41,828] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-01 09:47:41,830] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-01 09:47:41,832] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-01 09:47:41,833] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-01 09:47:41,836] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-01 09:47:41,838] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-01 09:47:41,840] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-01 09:47:41,841] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-01 09:47:41,844] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-01 09:47:41,851] [ForkServerProcess-4

[2026-04-01 09:47:59.375] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-01 09:47:59.669] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-01 09:47:59.922] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


KeyboardInterrupt: 

我不明白发生了什么，这段代码它不报错，但运行一个多小时也不结束，CPU占用也一直很低

In [15]:
emulator.down()

[2026-04-01 09:55:29,513]-[INFO]-[emulation.py:120]: Shutdown multiprocess cluster...
